# 08 · Which words of a fusion text did the AI write?

According to the GhostWriter paper (Section 3.2), a fusion text is its human source with up to 50% of the
sentences or paragraphs masked and rewritten by an LLM. In `train.csv` most fusion texts share a source group with
their human source (notebook 01), so a word-level comparison of the two shows which words were kept and which were
written by the AI. These word labels are the training target for a token-level tagger (a model that labels every
token instead of the whole text), the approach of SemEval-2024 Task 8, subtask C.

This notebook builds the labels, checks them against the paper's description, and looks at where the formatting
traces found in the EDA occur. Only `train.csv` is used.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
P = env.setup(seed=42)
ensure_data(P)

platform: local
    data: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data
 outputs: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs
   cache: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/cache
    repo: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection
  device: mps
data ready at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data


PosixPath('/Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data')

In [2]:
# Imports; train texts with their source groups; the human texts of every group
import re
import time
import numpy as np
import pandas as pd

from hwkit.split import load_split
from hwkit.spans import best_source, parse_spans, span_record
from hwkit.stack import SEAMS

NOTEBOOK_START = time.time()
SPAN_FILE = P.repo / "results" / "fusion_spans.csv.gz"
train = load_split(pd.read_csv(P.data / "train.csv"))
train["domain_lang"] = train["domain"] + " | " + train["lang"]
humans = train[train["label"] == "human"].groupby("group")["id"].apply(list)
text_of = train.set_index("id")["text"]
fusion = train[train["label"] == "fusion"].reset_index(drop=True)
fusion["has_source"] = fusion["group"].isin(humans.index)
print(f"fusion texts: {len(fusion)}, with a human text in their group: {fusion['has_source'].mean():.1%}")

fusion texts: 83682, with a human text in their group: 91.5%


**Findings.** 91.5% of the training fusion texts have a human text in their source group; the rest cannot be
labelled and are left out.

In [3]:
# Word-level diff of every fusion text against the most similar human text of its group; saved into the repo
t0 = time.time()
records = []
for _, r in fusion[fusion["has_source"]].iterrows():
    cands = humans[r["group"]]
    src = cands[best_source(r["text"], [text_of[i] for i in cands])]
    records.append({"id": r["id"], "source_id": src, **span_record((text_of[src], r["text"]))})
spans = pd.DataFrame(records).merge(fusion[["id", "fold", "domain_lang"]], on="id")
spans.to_csv(SPAN_FILE, index=False)
print(f"{len(spans)} fusion texts labelled in {(time.time() - t0) / 60:.1f} min; "
      f"{SPAN_FILE.name}: {SPAN_FILE.stat().st_size / 1e6:.1f} MB")

76603 fusion texts labelled in 3.1 min; fusion_spans.csv.gz: 3.0 MB


In [4]:
# Check against the paper: share of AI words and number of AI sections per fusion text, by domain
q = lambda p: (lambda s: s.quantile(p))
summary = spans.groupby("domain_lang").agg(texts=("id", "size"), ai_share_median=("ai_share", "median"),
                                           ai_share_p10=("ai_share", q(0.1)), ai_share_p90=("ai_share", q(0.9)),
                                           one_section=("n_spans", lambda s: (s == 1).mean()),
                                           no_section=("n_spans", lambda s: (s == 0).mean()))
summary.loc["all"] = [len(spans), spans["ai_share"].median(), spans["ai_share"].quantile(0.1),
                      spans["ai_share"].quantile(0.9), (spans["n_spans"] == 1).mean(), (spans["n_spans"] == 0).mean()]
summary.round(3)

,texts,ai_share_median,ai_share_p10,ai_share_p90,one_section,no_section
domain_lang,,,,,,
arxiv_papers | en-EN,656.0,0.250,0.127,0.431,0.165,0.003
blog_authorship_corpus | en-EN,11802.0,0.104,0.026,0.367,0.927,0.001
bundestag | de-DE,17334.0,0.267,0.062,0.480,0.774,0.001
cnn_news | en-EN,11306.0,0.172,0.092,0.359,0.156,0.000
euro_court_cases | en-EN,9256.0,0.218,0.070,0.449,0.344,0.004
gutenberg | de-DE,31.0,0.141,0.024,0.588,0.871,0.000
gutenberg | en-EN,246.0,0.234,0.069,0.526,0.695,0.000
house_of_commons | en-EN,7692.0,0.194,0.017,0.467,0.711,0.002
student_essays | en-EN,18280.0,0.293,0.005,0.484,0.776,0.011


**Findings.** The recovered sections match the paper's description: the median fusion text has 22% AI-written
words (10–90% range: 4–46%), close to the "up to 50%" of Section 3.2. Blogs, student essays, Bundestag and House of
Commons mostly have one AI section; CNN, arxiv and court cases usually have several, either because several
segments were masked or because the diff splits one rewritten section where the AI kept a few source phrases.
Only 0.4% of fusion texts show no AI section at all.

In [5]:
# Where the AI section starts (share of the text before it) and how many sections there are
print("AI sections per text:", spans["n_spans"].clip(upper=4).value_counts(normalize=True).sort_index()
      .rename(index={4: "4+"}).round(3).to_dict())
print("start of the first AI section, deciles of the position in the text:",
      spans["first_ai"].quantile(np.linspace(0.1, 0.9, 9)).round(2).tolist())

AI sections per text: {0: 0.004, 1: 0.643, 2: 0.182, 3: 0.072, '4+': 0.099}
start of the first AI section, deciles of the position in the text: [0.0, 0.08, 0.18, 0.27, 0.38, 0.48, 0.58, 0.68, 0.81]


**Findings.** The first AI section can start anywhere in the text (deciles evenly spread from 0 to 0.8), so a model
that reads only the beginning misses many of them. This is one more reason to read every chunk.

In [6]:
# One typical example: the fusion text with the median AI share and one section, the AI part between [[ ]]
ex = spans[spans["n_spans"] == 1].iloc[(spans.loc[spans["n_spans"] == 1, "ai_share"]
                                        - spans["ai_share"].median()).abs().argmin()]
s, e = parse_spans(ex["spans"])[0]
t = text_of[ex["id"]]
print(f"{ex['domain_lang']}, {ex['n_words']} words, AI share {ex['ai_share']:.2f}\n")
print("…" + t[max(0, s - 300):s] + "[[" + t[s:min(e, s + 400)] + (" … " if e > s + 400 else "") + "]]" + t[e:e + 300] + "…")

bundestag | de-DE, 695 words, AI share 0.22

…[[Werte Kolleginnen und Kollegen, wir stehen heute hier, um über einen Antrag zu debattieren, der nicht nur unsere politische Landschaft, sondern auch die Zukunft unseres Landes maßgeblich beeinflussen wird. Es geht um die Frage, wie wir den Herausforderungen des Klimawandels begegnen und welche Maßnahmen wir ergreifen, um unseren Planeten zu schützen. Doch ich frage Sie: Geht es bei diesem Antrag w … ]] Der Versuch der Beantwortung führt unweigerlich zur Gretchenfrage: Ist die Aussage, der Klimawandel sei hauptsächlich menschengemacht, denn wahr? Bislang ist das eine These, nicht mehr und nicht weniger, genauso gültig wie die These, dass der Klimawandel nicht menschengemacht sei. Aus Ihrer These b…


In [7]:
# Where do the formatting traces occur? Per 1000 characters in human texts, in the kept and AI parts of fusion texts,
# and right at the edges of AI sections (within 3 characters of a start or end, per 100 edges)
EDGE = 3
pats = {k: re.compile(p) for k, p in SEAMS.items()}
cnt = {k: {"kept": 0, "ai": 0, "edge": 0} for k in pats}
chars = {"kept": 0, "ai": 0}
edges = 0
for r in spans.itertuples():
    t, sp = text_of[r.id], parse_spans(r.spans)
    ai_chars = sum(e - s for s, e in sp)
    chars["ai"] += ai_chars
    chars["kept"] += len(t) - ai_chars
    edges += 2 * len(sp)
    for k, p in pats.items():
        for m in p.finditer(t):
            x = m.start()
            if any(abs(x - s) <= EDGE or abs(x - e) <= EDGE for s, e in sp):
                cnt[k]["edge"] += 1
            elif any(s <= x < e for s, e in sp):
                cnt[k]["ai"] += 1
            else:
                cnt[k]["kept"] += 1
hum = train.loc[train["label"] == "human", "text"]
pd.DataFrame({k: {"human texts": 1000 * hum.str.count(SEAMS[k]).sum() / hum.str.len().sum(),
                  "fusion, kept parts": 1000 * c["kept"] / chars["kept"],
                  "fusion, AI parts": 1000 * c["ai"] / chars["ai"],
                  "per 100 AI-section edges": 100 * c["edge"] / max(edges, 1)} for k, c in cnt.items()}).T.round(3)

,human texts,"fusion, kept parts","fusion, AI parts",per 100 AI-section edges
space_before_closing_quote,0.005,0.005,0.003,5.851
space_before_punct,0.627,0.466,0.009,0.636
double_space,0.000,0.000,0.000,0.000
no_space_after_sentence,0.008,0.005,0.002,0.001


**Findings.** A space before a closing quote occurs at about 6 of 100 AI-section edges but only 0.005 times per 1000
characters elsewhere: it is a splice mark left where the AI text was inserted. A space before punctuation appears in
human writing (0.5–0.6 per 1000 characters) but almost never inside AI-written parts (0.009): AI text is "too
clean". Double spaces do not occur at all.

In [8]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 3.8 min


[]